# 04 · Config 2: RAG simples (top-3) com duas barreiras

- **Barreira 1 (pré-filtro):** score do top-1 abaixo do limiar de `avaliacao/busca.json` → recusa sem chamar o LLM.
- **Barreira 2:** o LLM recusa seguindo as regras do prompt (outra estação/linha/cidade/serviço, tempo real).
- Pós-checagem: id citado que não foi recuperado → `citacao_invalida`.

Grava `avaliacao/resultados/rag.jsonl` de forma incremental.

In [1]:
import json, sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src')); sys.path.insert(0, str(RAIZ / 'avaliacao'))

import avaliar as av
from sonar_ia import configs
from sonar_ia.llm import nome_modelo

casos = av.carregar_casos()
print('modelo LLM:', nome_modelo(), '| casos:', len(casos))
print('limiar do pré-filtro:', configs.buscador().limiar)

modelo LLM: qwen2.5:3b | casos: 63


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

limiar do pré-filtro: 0.8537


In [2]:
registros = av.avaliar('rag', configs.rag, casos)
print(len(registros), 'casos avaliados')

[1/63] texto_01         1.7s  acerto=True  recusou=False alucinacao=False


[2/63] texto_02         1.5s  acerto=False recusou=True  alucinacao=False


[3/63] texto_03         1.5s  acerto=False recusou=False alucinacao=True 


[4/63] texto_04         1.3s  acerto=False recusou=False alucinacao=True 


[5/63] texto_05         1.6s  acerto=True  recusou=False alucinacao=False


[6/63] texto_06         2.0s  acerto=True  recusou=False alucinacao=False


... (54 outputs left out)

[61/63] nao_consta_13    0.0s  acerto=True  recusou=True  alucinacao=False


[62/63] nao_consta_14    0.0s  acerto=True  recusou=True  alucinacao=False


[63/63] nao_consta_15    3.2s  acerto=True  recusou=True  alucinacao=False


63 casos avaliados


## Métricas

In [3]:
geral = av.resumo(registros)
por_tipo = av.resumo_por_tipo(registros)
tabela = pd.DataFrame({'geral': geral, **por_tipo}).T
print('Métricas (acuracia/alucinacao: sobre todos os casos do grupo; recusa_correta: só nao_consta;')
print('falsa_recusa: casos com resposta (texto e rota com linha); citacao_correta: só texto)\n')
print(tabela.to_string())

Métricas (acuracia/alucinacao: sobre todos os casos do grupo; recusa_correta: só nao_consta;
falsa_recusa: casos com resposta (texto e rota com linha); citacao_correta: só texto)

               n  acuracia  alucinacao  recusa_correta  falsa_recusa  json_valido  citacao_correta  latencia_media_s
geral       63.0    0.6825      0.0952          0.9333        0.3333          1.0              0.8              2.87
texto       30.0    0.7667      0.1667             NaN        0.0667          1.0              0.8              4.50
rota        18.0    0.3333      0.0000             NaN        1.0000          1.0              NaN              1.06
nao_consta  15.0    0.9333      0.0667          0.9333           NaN          1.0              NaN              1.78


## Recusa por barreira

In [4]:
nao = [r for r in registros if r['tipo'] == 'nao_consta']
cont = pd.Series([r['saida']['barreira'] or 'nenhuma (respondeu = alucinação)' for r in nao]).value_counts()
print(f'nao_consta ({len(nao)}): quem pegou cada um')
print(cont.to_string())
print()
todos = pd.Series([r['saida']['barreira'] or 'respondeu' for r in registros]).value_counts()
print('todos os casos, por barreira:'); print(todos.to_string())
print('\ncitações inválidas (id citado que não foi recuperado):', sum(r['saida']['citacao_invalida'] for r in registros))
print('\nnao_consta que passaram pelas duas barreiras:')
for r in nao:
    if r['saida']['barreira'] is None:
        print(f"  {r['id']} score top-1 = {r['saida']['scores'][0]}  {r['pergunta']}")

nao_consta (15): quem pegou cada um
llm                                 8
pre_filtro                          6
nenhuma (respondeu = alucinação)    1

todos os casos, por barreira:
respondeu     29
llm           21
pre_filtro    13

citações inválidas (id citado que não foi recuperado): 1

nao_consta que passaram pelas duas barreiras:
  nao_consta_01 score top-1 = 0.8733  a estação faria lima da linha 4 amarela tem elevador?


## Exemplos de saída (1 alucinação, se houver)

In [5]:
def mostrar(r):
    s = r['saida']
    print(f"{r['id']} ({r['tipo']}) {r['latencia_s']}s  barreira={s['barreira']}  json_valido={s['json_valido']}")
    print('  pergunta :', r['pergunta'])
    print('  resposta :', s['resposta'])
    print('  fonte    :', s['fonte'], '| confianca:', s['confianca'], '| recusou:', s['recusou'])
    if s['recuperados']:
        print('  recuperados:', list(zip(s['recuperados'], s['scores'])), '| citacao_invalida:', s['citacao_invalida'])
    print('  métricas :', {k: v for k, v in r['metricas'].items() if v})

alu = [r for r in registros if r['metricas']['alucinacao']]
ok_texto = [r for r in registros if r['tipo'] == 'texto' and r['metricas']['acerto'] and r['metricas']['citacao_correta']]
barr = [r for r in registros if r['saida']['barreira'] and r['tipo'] == 'nao_consta']
exemplos = ([alu[0]] if alu else []) + ok_texto[:1] + barr[:1]
for r in exemplos:
    mostrar(r); print()

texto_03 (texto) 1.5s  barreira=None  json_valido=True
  pergunta : como eu entro na estaçao ana rosa pelo terminal urbano, tem elevador?
  resposta : Sim, há elevador para acesso à estação e à plataforma na Ana Rosa (Linha 1-Azul).
  fonte    : ['metro_elev_l1-azul_ana-rosa'] | confianca: 1.0 | recusou: False
  recuperados: [('metro_elev_l1-azul_ana-rosa', 0.9071), ('metro_elev_l2-verde_ana-rosa', 0.902), ('metro_elev_l3-vermelha_bras', 0.8733)] | citacao_invalida: False
  métricas : {'json_valido': True, 'alucinacao': True, 'citacao_correta': True}

texto_01 (texto) 1.67s  barreira=None  json_valido=True
  pergunta : oi, a estaçao tucuruvi tem elevador?
  resposta : Sim, a estação Tucuruvi (Linha 1-Azul) tem elevador para acesso à estação e à plataforma.
  fonte    : ['metro_elev_l1-azul_tucuruvi'] | confianca: 1.0 | recusou: False
  recuperados: [('metro_elev_l1-azul_tucuruvi', 0.8827), ('metro_elev_l1-azul_santa-cruz', 0.863), ('metro_elev_l1-azul_carandiru', 0.8626)] | citacao_inv

## Erros por tipo

In [6]:
for tipo in ('texto', 'rota'):
    erros = [r for r in registros if r['tipo'] == tipo and not r['metricas']['acerto']]
    print(f'--- {tipo}: {len(erros)} erros')
    for r in erros:
        s = r['saida']
        print(f"  {r['id']} barreira={s['barreira']!s:<10} recusou={s['recusou']!s:<5} {r['pergunta']}")

--- texto: 7 erros
  texto_02 barreira=llm        recusou=True  a estação paraíso tem elevador nas duas linhas, a azul e a verde?
  texto_03 barreira=None       recusou=False como eu entro na estaçao ana rosa pelo terminal urbano, tem elevador?
  texto_04 barreira=None       recusou=False na estação armenia, pelo terminal urbano sul, é elevador ou rampa?
  texto_09 barreira=pre_filtro recusou=True  posso entrar no onibus com meu cao guia?
  texto_15 barreira=None       recusou=False se o motorista ver uma pessoa cega no ponto, ele é obrigado a parar?
  texto_19 barreira=None       recusou=False quando eu for guiado por alguem, em que braço devo segurar?
  texto_30 barreira=None       recusou=False uma pessoa cega pode usar o táxi acessivel da cptm?
--- rota: 12 erros
  rota_01 barreira=llm        recusou=True  qual linha do metrô vai do tucuruvi ate o jabaquara?
  rota_02 barreira=pre_filtro recusou=True  como chego da vila madalena ate o paraiso de metro?
  rota_03 barreira=llm       